# LangGraph RAG Application - Demo Notebook

This notebook demonstrates and tests all functionality of the LangGraph RAG (Retrieval-Augmented Generation) application for codebase Q&A.

## Architecture Overview

The application uses:
- **Document Loading**: Load Python files with metadata enrichment
- **Text Splitting**: Code-aware chunking using LangChain splitters
- **Vector Store**: ChromaDB with OpenAI embeddings
- **LangGraph Workflow**: Self-correcting retrieval with query rewriting
- **LLM**: OpenAI GPT-4 for grading and generation

## Section 1: Setup and Configuration

First, we'll set up the environment and import all necessary modules.

In [ ]:
# Install dependencies (uncomment if needed)
# !pip install langchain langchain-community langchain-openai langchain-chroma langgraph python-dotenv

In [ ]:
import os
import sys
from pathlib import Path
from dotenv import load_dotenv

# Add src directory to Python path
src_path = Path.cwd() / 'src'
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

# Load environment variables
load_dotenv()

# Verify OpenAI API key is set
if not os.getenv('OPENAI_API_KEY'):
    raise ValueError("OPENAI_API_KEY not found in .env file")

print("✓ Environment loaded successfully")
print(f"✓ Working directory: {Path.cwd()}")
print(f"✓ OpenAI API key is set: {os.getenv('OPENAI_API_KEY')[:10]}...")

In [ ]:
# Import all necessary modules
from loader.code_loader import CodeLoader
from processing.text_splitter import get_code_splitter, split_documents
from vectorstore.chroma_store import ChromaStore
from graph.state import GraphState
from graph.nodes import retrieve, grade_documents, generate, transform_query, get_llm
from graph.workflow import create_workflow, decide_to_generate

print("✓ All modules imported successfully")

## Section 2: Document Loading

Load Python files from the test_data directory using the `CodeLoader` class.

In [ ]:
# Initialize CodeLoader with test data
test_data_path = Path.cwd() / 'test_data'
print(f"Loading code from: {test_data_path}")

loader = CodeLoader(str(test_data_path), file_extensions=['.py'])
documents = loader.load_documents()

print(f"\n✓ Loaded {len(documents)} documents")

In [ ]:
# Display statistics about loaded documents
stats = loader.get_stats(documents)

print("Document Statistics:")
print(f"  Total files: {stats['total_files']}")
print(f"  Total characters: {stats['total_characters']:,}")
print(f"  Average file size: {stats['average_file_size']:,} chars")
print(f"  Files by extension: {stats['files_by_extension']}")

In [ ]:
# Display sample content from first document
if documents:
    doc = documents[0]
    print(f"\nSample Document:")
    print(f"  File: {doc.metadata.get('relative_path', 'unknown')}")
    print(f"  Extension: {doc.metadata.get('file_extension', 'unknown')}")
    print(f"  Source: {doc.metadata.get('source', 'unknown')}")
    print(f"  Content length: {len(doc.page_content)} chars")
    print(f"\nFirst 500 characters:")
    print(doc.page_content[:500])

## Section 3: Text Splitting

Split documents into chunks using code-aware text splitting.

In [ ]:
# Configure chunk sizes
CHUNK_SIZE = 1000
CHUNK_OVERLAP = 200

print(f"Splitting configuration:")
print(f"  Chunk size: {CHUNK_SIZE}")
print(f"  Chunk overlap: {CHUNK_OVERLAP}")

# Split documents
split_docs = split_documents(documents, chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)

print(f"\n✓ Created {len(split_docs)} chunks from {len(documents)} documents")
print(f"  Average chunks per document: {len(split_docs) / len(documents):.1f}")

In [ ]:
# Display sample chunks
print("\nSample Chunks:")
for i, chunk in enumerate(split_docs[:3], 1):
    print(f"\n--- Chunk {i} ---")
    print(f"  File: {chunk.metadata.get('relative_path', 'unknown')}")
    print(f"  Chunk ID: {chunk.metadata.get('chunk_id', 'N/A')}")
    print(f"  Start index: {chunk.metadata.get('start_index', 'N/A')}")
    print(f"  Length: {len(chunk.page_content)} chars")
    print(f"  Preview: {chunk.page_content[:200]}...")

## Section 4: Vector Store Operations

Create a ChromaDB vector store and test similarity search.

In [ ]:
# Initialize ChromaStore with test collection
chroma_store = ChromaStore(
    persist_directory='./test_chroma_db',
    collection_name='demo_collection',
    embedding_model='text-embedding-3-small'
)

print("Creating vector store...")
vectorstore = chroma_store.create(split_docs)
print("✓ Vector store created successfully")

In [ ]:
# Display vector store statistics
store_stats = chroma_store.get_stats()

print("\nVector Store Statistics:")
print(f"  Collection: {store_stats['collection_name']}")
print(f"  Documents: {store_stats['document_count']}")
print(f"  Directory: {store_stats['persist_directory']}")

In [ ]:
# Test direct similarity search
test_query = "How does the Calculator add method work?"
print(f"Test query: '{test_query}'\n")

results = chroma_store.query(test_query, k=3)

print(f"Retrieved {len(results)} documents:\n")
for i, doc in enumerate(results, 1):
    print(f"--- Result {i} ---")
    print(f"  File: {doc.metadata.get('relative_path', 'unknown')}")
    print(f"  Content preview: {doc.page_content[:200]}...\n")

## Section 5: Individual Node Testing

Test each node function independently before running the full workflow.

### 5.1: Test Retrieve Node

In [ ]:
# Create retriever
retriever = chroma_store.as_retriever()

# Test retrieve function
test_state = {
    "question": "What error handling is available?",
    "documents": [],
    "generation": "",
    "relevance_score": "",
    "retry_count": 0
}

print("Testing retrieve node...\n")
result = retrieve(test_state, retriever)

print(f"\n✓ Retrieved {len(result['documents'])} documents")
for i, doc in enumerate(result['documents'], 1):
    print(f"  {i}. {doc.metadata.get('relative_path', 'unknown')}")

### 5.2: Test Grade Documents Node

In [ ]:
# Test grade_documents function
test_state_grade = {
    "question": "What error handling is available?",
    "documents": result['documents'],
    "generation": "",
    "relevance_score": "",
    "retry_count": 0
}

print("Testing grade_documents node...\n")
grading_result = grade_documents(test_state_grade)

print(f"\n✓ Grading complete")
print(f"  Relevance score: {grading_result['relevance_score']}")
print(f"  Filtered documents: {len(grading_result['documents'])}")

### 5.3: Test Generate Node

In [ ]:
# Test generate function
test_state_gen = {
    "question": "What error handling is available?",
    "documents": grading_result['documents'],
    "generation": "",
    "relevance_score": "yes",
    "retry_count": 0
}

print("Testing generate node...\n")
generation_result = generate(test_state_gen)

print(f"\n✓ Generation complete\n")
print("Generated Answer:")
print("="*80)
print(generation_result['generation'])
print("="*80)

### 5.4: Test Transform Query Node

In [ ]:
# Test transform_query function
test_state_transform = {
    "question": "How do I add numbers?",
    "documents": [],
    "generation": "",
    "relevance_score": "no",
    "retry_count": 0
}

print("Testing transform_query node...\n")
transform_result = transform_query(test_state_transform)

print(f"\n✓ Query transformation complete")
print(f"  New retry count: {transform_result['retry_count']}")

## Section 6: Full Workflow Execution

Create and execute the complete LangGraph workflow with self-correcting retrieval.

In [ ]:
# Create the workflow
print("Creating LangGraph workflow...")
app = create_workflow(retriever)
print("✓ Workflow created successfully\n")

# Display workflow structure
print("Workflow structure:")
print("  1. retrieve → Fetch relevant documents")
print("  2. grade_documents → Check relevance")
print("  3. Decision: relevant? → generate : transform_query")
print("  4. transform_query → retrieve (retry loop)")
print("  5. generate → END")

In [ ]:
# Run a simple query through the full workflow
question = "How does the Calculator class track operation history?"

print(f"Question: {question}\n")
print("="*80)
print("WORKFLOW EXECUTION LOG")
print("="*80)

# Execute workflow
result = app.invoke({
    "question": question,
    "retry_count": 0
})

print("\n" + "="*80)
print("FINAL ANSWER")
print("="*80)
print(result['generation'])
print("\n" + "="*80)
print(f"Documents used: {len(result.get('documents', []))}")
print(f"Retry count: {result.get('retry_count', 0)}")

## Section 7: Interactive Querying

Test multiple queries including edge cases and vague questions to demonstrate self-correcting retrieval.

In [ ]:
# Define test queries
test_queries = [
    "What payment methods are supported in the payment processing function?",
    "How does the Calculator handle division by zero?",
    "What custom exception classes are defined for API errors?",
    "Show me the validation logic",  # Vague query - should trigger rewrite
    "How do I handle errors?",  # Vague query - should trigger rewrite
]

# Execute each query
for i, query in enumerate(test_queries, 1):
    print("\n" + "#"*80)
    print(f"Query {i}: {query}")
    print("#"*80 + "\n")
    
    result = app.invoke({
        "question": query,
        "retry_count": 0
    })
    
    print("\nANSWER:")
    print("-"*80)
    print(result['generation'])
    print("-"*80)
    print(f"Retries: {result.get('retry_count', 0)} | Documents: {len(result.get('documents', []))}")
    print("\n")

### Query Rewriting Demonstration

Test a deliberately vague query to see the query rewriting mechanism in action.

In [ ]:
# Vague query that should trigger query rewriting
vague_query = "Tell me about calculations"

print(f"Testing query rewriting with vague query: '{vague_query}'\n")
print("="*80)
print("Watch for TRANSFORM QUERY messages if documents are not relevant")
print("="*80 + "\n")

result = app.invoke({
    "question": vague_query,
    "retry_count": 0
})

print("\n" + "="*80)
print("RESULT")
print("="*80)
print(result['generation'])
print("\n" + "="*80)
print(f"Query was rewritten {result.get('retry_count', 0)} time(s)")

### Custom Query Testing

Define your own query to test the system.

In [ ]:
# Enter your custom query here
custom_query = "What functions are available for processing payments?"

print(f"Custom Query: {custom_query}\n")
print("="*80)

result = app.invoke({
    "question": custom_query,
    "retry_count": 0
})

print("\nANSWER:")
print("="*80)
print(result['generation'])
print("="*80)
print(f"\nMetadata:")
print(f"  Retries: {result.get('retry_count', 0)}")
print(f"  Documents: {len(result.get('documents', []))}")
print(f"  Relevance: {result.get('relevance_score', 'N/A')}")

## Section 8: Cleanup

Clean up test resources and delete the test collection.

In [ ]:
# Display collection info before deletion
stats = chroma_store.get_stats()
print("Current collection:")
print(f"  Name: {stats['collection_name']}")
print(f"  Documents: {stats['document_count']}")
print(f"  Directory: {stats['persist_directory']}")

In [ ]:
# Delete the test collection
print("\nDeleting test collection...")
chroma_store.delete_collection()
print("✓ Test collection deleted")

In [ ]:
# Optionally, remove the test directory
import shutil

test_db_path = Path('./test_chroma_db')
if test_db_path.exists():
    print(f"Removing test database directory: {test_db_path}")
    shutil.rmtree(test_db_path)
    print("✓ Directory removed")
else:
    print("Directory already removed or doesn't exist")

## Summary

This notebook demonstrated:

1. **Document Loading**: Loading Python files with metadata enrichment
2. **Text Splitting**: Code-aware chunking for optimal retrieval
3. **Vector Store**: Creating and querying ChromaDB collections
4. **Individual Nodes**: Testing each workflow component independently
5. **Full Workflow**: End-to-end question answering with self-correction
6. **Query Rewriting**: Automatic query improvement when documents aren't relevant
7. **Interactive Testing**: Multiple example queries with detailed logging
8. **Cleanup**: Proper resource management

### Key Features Demonstrated

- **Self-Correcting Retrieval**: The system automatically rewrites queries when retrieved documents aren't relevant
- **Document Grading**: LLM-based relevance scoring ensures only pertinent documents are used
- **Code-Aware Processing**: Specialized text splitting preserves code structure
- **Metadata Tracking**: Rich metadata for debugging and understanding retrieval

### Next Steps

- Index your own codebase by changing the `test_data_path`
- Adjust chunk sizes and overlap for your use case
- Experiment with different queries and observe the workflow
- Modify the prompts in `src/graph/nodes.py` for custom behavior
- Add more node types or conditional edges to the workflow